# Paper Experiments: AIE & GAIE (ablations, distributions, efficiency)

Companion notebook to `full-run-unlearn-<dataset>.ipynb`. That notebook produces the **main
table** (Retrain / AIE / CIE / GAIE / HIE under the adversarial-attack protocol). This one
produces everything else a publication needs, mirroring the research questions of
*"Pre-training for Recommendation Unlearning"* (UnlearnRec, SIGIR'25):

| Section | Paper RQ | Experiment |
|---|---|---|
| EXP-1 | RQ3 | Ablation study: remove L_u / L_p / L_rec / fine-tuning for AIE and GAIE |
| EXP-2 | RQ4 | Link-prediction score distributions (positive / negative / adversarial edges) |
| EXP-3 | RQ5 | Efficiency: wall-clock unlearning time vs. retraining |
| EXP-4 | — | Random-request unlearning (the privacy / right-to-be-forgotten scenario) |
| EXP-5 | — | Loss-weight sensitivity (optional, flag-gated) |
| EXP-6 | — | Multi-seed robustness (optional, flag-gated) |

**Protocol.** Backbone LightGCN is trained on the **attacked graph** (clean edges + injected
least-probable adversarial edges), per UnlearnRec Sec. 4.1.4; the unlearning target is the
injected edge set. The exact-unlearning ground truth is the same backbone retrained from
scratch on the residual (clean) graph. If checkpoints from the full-run notebook exist in
`./ckpt/`, they are reused; otherwise they are trained here.

**Runtime warning.** EXP-1 runs 9 unlearn+finetune pipelines. On ML-1M each is a few minutes
on a Kaggle T4/P100; on Gowalla/Yelp trim the variant lists or run EXP-1 only for the dataset
you feature in the ablation table (the paper ablates on two datasets only).

### Kaggle Setup
Enable **GPU accelerator** (Settings -> Accelerator -> GPU).

---
## 0. Environment

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from Utils.utils import innerProduct, cal_mi_metrics, print_args
from models.Model import LightGCN, AIE, CIE, GAIE, HIE

print("All imports successful!")

In [ ]:
import torch as t
import numpy as np
import random
import time

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")
    print(f"Memory: {t.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. Configuration

In [ ]:
import json
import time
import matplotlib.pyplot as plt

DATASET = 'ml1m'   # 'ml1m' | 'gowalla' | 'yelp2018'

CFG = {
    'ml1m': dict(
        data='ml1m', pretrain_batch=2048, pretrain_epoch=50, adv_method='lightgcn0.5',
        pretrain_drop_rate=0.1, attacked_ckpt='./ckpt/pretrain_adv',
        aie_ft=dict(epoch=20, unlearn_wei=0.1, align_wei=0.05),
        gaie_ft=dict(epoch=15, unlearn_wei=0.15, align_wei=0.1),
    ),
    'gowalla': dict(
        data='gowalla', pretrain_batch=4096, pretrain_epoch=200, adv_method='lightgcn0.5',
        pretrain_drop_rate=0.2, attacked_ckpt='./ckpt/pretrain_gowalla_adv',
        aie_ft=dict(epoch=20, unlearn_wei=0.1, align_wei=0.05),
        gaie_ft=dict(epoch=15, unlearn_wei=0.15, align_wei=0.1),
    ),
    'yelp2018': dict(
        data='yelp2018', pretrain_batch=4096, pretrain_epoch=350, adv_method='lightgcn',
        pretrain_drop_rate=0.1, attacked_ckpt='./ckpt/pretrain_yelp2018_adv',
        aie_ft=dict(epoch=20, unlearn_wei=0.2, align_wei=0.01),
        gaie_ft=dict(epoch=15, unlearn_wei=0.15, align_wei=0.1),
    ),
}[DATASET]

args.data = CFG['data']
args.model = 'lightgcn'
args.gpu = '0'
args.seed = 1234
args.lr = 1e-3
args.latdim = 128
args.gnn_layer = 3
args.reg = 1e-7
args.topk = 20
args.tst_epoch = 3
args.tst_bat = 256
args.decay = 1.0
args.bpr_wei = 1.0
args.adv_method = CFG['adv_method']

ATTACKED_CKPT = CFG['attacked_ckpt']
CLEAN_CKPT = './ckpt/retrain_ref'   # same path the full-run notebook uses

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu


def reset_seeds(seed=None):
    seed = seed if seed is not None else args.seed
    t.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    t.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)


def apply_args(d):
    for k, v in d.items():
        setattr(args, k, v)


reset_seeds()
print_args(args)

---
## 2. Backbones (train or load)

- **Attacked backbone** — the model to be unlearned (trained on clean + adversarial edges).
- **Retrain reference** — exact-unlearning ground truth (trained on clean edges only).

In [ ]:
from training.pretrain_lightgcn import Coach as PretrainCoach

# ---- attacked backbone -------------------------------------------------
args.adversarial_attack = True
args.epoch = CFG['pretrain_epoch']
args.batch = CFG['pretrain_batch']
args.save_path = ATTACKED_CKPT
reset_seeds()

handler_atk = DataHandler()
handler_atk.load_data(drop_rate=0.0, adv_attack=True)
print(f"Users: {args.user}, Items: {args.item}, "
      f"injected adversarial edges: {len(handler_atk.adv_edges[0])}")

if not os.path.exists(ATTACKED_CKPT + '.mod'):
    _t0 = time.time()
    PretrainCoach(handler_atk).run()
    print(f"Attacked backbone trained in {time.time() - _t0:.1f}s")
else:
    print(f"Reusing attacked backbone checkpoint {ATTACKED_CKPT}.mod")

attacked_model = t.load(ATTACKED_CKPT + '.mod', weights_only=False)['model'].cuda()
attacked_model.eval()
_pc = PretrainCoach(handler_atk)
res_atk = _pc.tst_epoch(attacked_model)
print(f"[Attacked backbone] Recall@{args.topk}: {res_atk['Recall']:.4f}, "
      f"NDCG@{args.topk}: {res_atk['NDCG']:.4f}")

In [ ]:
# ---- retrain reference (ground truth) ----------------------------------
args.adversarial_attack = False
args.save_path = CLEAN_CKPT
reset_seeds()

handler_clean = DataHandler()
handler_clean.load_data(drop_rate=0.0, adv_attack=False)

retrain_time = None
if not os.path.exists(CLEAN_CKPT + '.mod'):
    _t0 = time.time()
    PretrainCoach(handler_clean).run()
    retrain_time = time.time() - _t0
    print(f"Retrain reference trained in {retrain_time:.1f}s")
else:
    print(f"Reusing retrain-reference checkpoint {CLEAN_CKPT}.mod "
          f"(retrain wall-clock unavailable this session; take it from the full-run log)")

retrain_model = t.load(CLEAN_CKPT + '.mod', weights_only=False)['model'].cuda()
retrain_model.eval()
_pc = PretrainCoach(handler_clean)
res_ret = _pc.tst_epoch(retrain_model)
print(f"[Retrain reference] Recall@{args.topk}: {res_ret['Recall']:.4f}, "
      f"NDCG@{args.topk}: {res_ret['NDCG']:.4f}")
clean_adj = handler_clean.ts_ori_adj
del _pc

---
## 3. Shared harness

`run_variant` runs one (pipeline, variant) unlearning experiment end to end — unlearn stage,
optional fine-tune stage, evaluation on the best checkpoint — and appends a result row.
Score-collection helpers extract the raw link-prediction scores used by EXP-2.

In [ ]:
from unlearning.aie_unlearn import Coach as AIECoach
from unlearning.gaie_unlearn import Coach as GAIECoach

COACHES = {'aie': AIECoach, 'gaie': GAIECoach}

BASE = {
    'aie': dict(
        epoch=30, lr=1e-3, batch=4096, pretrain_drop_rate=CFG['pretrain_drop_rate'],
        test_drop_rate=0.003, sim_epoch=10, tst_epoch=5, bpr_wei=1.0,
        unlearn_wei=0.3, align_wei=0.1, align_temp=10.0, align_type='v2', unlearn_type='v1',
        overall_withdraw_rate=0.1, withdraw_rate_init=1, hyper_temp=1.0, unlearn_ssl=1e-3,
        layer_mlp=2, leaky=0.99, act='leaky', unlearn_layer=0, perf_degrade=0.3,
    ),
    'gaie': dict(
        epoch=30, lr=1e-3, batch=4096, pretrain_drop_rate=CFG['pretrain_drop_rate'],
        test_drop_rate=0.003, sim_epoch=10, tst_epoch=5, bpr_wei=1.0,
        unlearn_wei=0.3, align_wei=0.1, rec_wei=0.03, align_temp=10.0, align_type='v2',
        unlearn_type='v1', overall_withdraw_rate=0.1, withdraw_rate_init=1, hyper_temp=1.0,
        unlearn_ssl=1e-3, layer_mlp=2, leaky=0.99, act='leaky', unlearn_layer=0,
        perf_degrade=0.5,
    ),
}
FT_OVERRIDES = {'aie': CFG['aie_ft'], 'gaie': CFG['gaie_ft']}

exp_results = []
scores_store = {}


def _sample_negs(handler, n):
    rows, cols = handler.ori_trn_mat.row, handler.ori_trn_mat.col
    edge_set = set(zip(rows.tolist(), cols.tolist()))
    nr, nc = [], []
    while len(nr) < n:
        i, j = np.random.randint(args.user), np.random.randint(args.item)
        if (i, j) not in edge_set:
            edge_set.add((i, j))
            nr.append(i)
            nc.append(j)
    return nr, nc


def _scores_from_embeds(u_emb, i_emb, handler):
    del_u, del_i = handler.dropped_edges
    pk_u, pk_i = handler.picked_edges
    k = len(del_u)
    idx = np.random.permutation(len(pk_u))[:k]
    pu = t.tensor(pk_u).long()[idx]
    pi = t.tensor(pk_i).long()[idx]
    nr, nc = _sample_negs(handler, k)
    return {
        'adv': innerProduct(u_emb[del_u], i_emb[del_i]).detach().cpu().numpy(),
        'pos': innerProduct(u_emb[pu], i_emb[pi]).detach().cpu().numpy(),
        'neg': innerProduct(u_emb[nr], i_emb[nc]).detach().cpu().numpy(),
    }


def gather_scores_ie(model, handler):
    with t.no_grad():
        u_emb, i_emb = model.outforward(handler.ts_ori_adj, handler.ts_pk_adj,
                                        handler.mask, handler.ts_drp_adj)
    return _scores_from_embeds(u_emb, i_emb, handler)


def gather_scores_plain(model, handler, adj):
    with t.no_grad():
        u_emb, i_emb = model.forward(adj, keepRate=1.0)
    return _scores_from_embeds(u_emb, i_emb, handler)


def run_variant(pipeline, variant, overrides=None, do_finetune=True,
                adversarial=True, trained_ckpt=None, collect_scores_to=None):
    Coach = COACHES[pipeline]
    overrides = overrides or {}
    tag = f"{pipeline}_{variant}" + ('' if adversarial else '_rand')
    print('\n' + '=' * 100)
    print(f"RUNNING VARIANT: {tag}  (finetune={do_finetune}, adversarial={adversarial})")
    print('=' * 100)

    # ---- unlearn stage ----
    args.adversarial_attack = adversarial
    apply_args(BASE[pipeline])
    apply_args(overrides)
    args.fineTune = False
    args.trained_model = trained_ckpt or ATTACKED_CKPT
    args.save_path = f'./ckpt/exp_{tag}'
    reset_seeds(overrides.get('seed'))

    handler_u = DataHandler()
    handler_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=adversarial)
    coach = Coach(handler_u)
    _t0 = time.time()
    coach.run()
    t_unlearn = time.time() - _t0

    # ---- fine-tune stage ----
    t_ft = 0.0
    if do_finetune:
        args.fineTune = True
        args.model_2_finetune = args.save_path
        apply_args(FT_OVERRIDES[pipeline])
        # keep ablated weights ablated in the fine-tune stage too
        apply_args({k: v for k, v in overrides.items()
                    if k in ('unlearn_wei', 'align_wei', 'rec_wei')})
        args.save_path = f'./ckpt/exp_{tag}_ft'
        reset_seeds(overrides.get('seed'))
        handler_f = DataHandler()
        handler_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=adversarial)
        coach = Coach(handler_f)
        _t0 = time.time()
        coach.run()
        t_ft = time.time() - _t0

    # ---- evaluate best checkpoint ----
    handler_e = DataHandler()
    handler_e.load_data(drop_rate=args.test_drop_rate, adv_attack=adversarial)
    sp = args.save_path if args.save_path.endswith('.mod') else args.save_path + '.mod'
    if os.path.exists(sp):
        coach.model = t.load(sp, weights_only=False)['model'].cuda()
    coach.handler = handler_e
    metrics = coach.tst_epoch(coach.model)
    mi = coach.test_unlearn(coach.model, prefix=f'[{tag}] final')

    if collect_scores_to is not None:
        scores_store[collect_scores_to] = gather_scores_ie(coach.model, handler_e)

    row = dict(pipeline=pipeline, variant=variant, adversarial=adversarial,
               Recall=metrics['Recall'], NDCG=metrics['NDCG'],
               MI_BF=mi['mi_bf'], MI_NG=mi['mi_ng'],
               BeforeProb=mi['avg_before_prob'], AfterProb=mi['avg_after_prob'],
               NegProb=mi['avg_neg_prob'], UnlearnTime=t_unlearn, FinetuneTime=t_ft)
    exp_results.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

    del coach, handler_u, handler_e
    if do_finetune:
        del handler_f
    t.cuda.empty_cache()
    return row

---
## EXP-1 — Ablation study (paper RQ3)

Loss-component knockouts, mirroring UnlearnRec Table 3:

| Variant | Meaning |
|---|---|
| `full` | complete pipeline (unlearn + fine-tune) |
| `no_Lu` | unlearning loss removed (`unlearn_wei = 0`) |
| `no_Lp` | preservation/alignment loss removed (`align_wei = 0`) |
| `no_Lrec` | **GAIE only** — VGAE reconstruction + KL removed (`rec_wei = 0`) |
| `no_finetune` | pre-trained influence encoder applied directly, no fine-tune stage |

Note: GAIE folds the KL term into the reconstruction loss with a fixed 0.01 factor inside
`GAIE.cal_reconstruction_loss`, so `rec_wei = 0` removes the whole variational objective.
Ablating the KL term alone requires exposing that constant as an arg (see `models/Model.py`).

In [ ]:
AIE_VARIANTS = [
    ('full',        {},                    True),
    ('no_Lu',       {'unlearn_wei': 0.0}, True),
    ('no_Lp',       {'align_wei': 0.0},   True),
    ('no_finetune', {},                    False),
]

for variant, overrides, ft in AIE_VARIANTS:
    run_variant('aie', variant, overrides, do_finetune=ft,
                collect_scores_to=('AIE' if variant == 'full' else None))

In [ ]:
GAIE_VARIANTS = [
    ('full',        {},                    True),
    ('no_Lu',       {'unlearn_wei': 0.0}, True),
    ('no_Lp',       {'align_wei': 0.0},   True),
    ('no_Lrec',     {'rec_wei': 0.0},     True),
    ('no_finetune', {},                    False),
]

for variant, overrides, ft in GAIE_VARIANTS:
    run_variant('gaie', variant, overrides, do_finetune=ft,
                collect_scores_to=('GAIE' if variant == 'full' else None))

In [ ]:
print(f"{'pipeline':<8} {'variant':<12} {'Recall@20':>10} {'NDCG@20':>9} "
      f"{'MI-BF':>8} {'MI-NG':>8} {'P(after)':>9} {'P(neg)':>8}")
print('-' * 78)
for r in exp_results:
    if not r['adversarial']:
        continue
    print(f"{r['pipeline']:<8} {r['variant']:<12} {r['Recall']:>10.4f} {r['NDCG']:>9.4f} "
          f"{r['MI_BF']:>8.4f} {r['MI_NG']:>8.4f} {r['AfterProb']:>9.4f} {r['NegProb']:>8.4f}")

os.makedirs('./logs', exist_ok=True)
with open(f'./logs/ablation_{DATASET}.json', 'w') as fs:
    json.dump(exp_results, fs, indent=2)
print(f"\nSaved to ./logs/ablation_{DATASET}.json")

---
## EXP-2 — Score distribution study (paper RQ4)

Distributions of link-prediction scores for positive, negative, and adversarial (unlearned)
edges — the analogue of UnlearnRec Figs. 2-3. Four panels: the attacked backbone before
unlearning, AIE, GAIE, and the retrained ground truth. Dashed lines mark per-group means.

Reading guide: before unlearning, adversarial edges should sit near positive edges (they were
trained as positives). A good unlearner moves them to — or below — the negative-edge
distribution, matching the retrained model's shape.

In [ ]:
# 'Before' and 'Retrain' panels (AIE/GAIE panels were collected during EXP-1 'full' runs)
args.adversarial_attack = True
handler_dist = DataHandler()
handler_dist.load_data(drop_rate=0.003, adv_attack=True)   # rate ignored in adv mode

reset_seeds()
scores_store['Before unlearning'] = gather_scores_plain(
    attacked_model, handler_dist, handler_dist.ts_ori_adj)
scores_store['Retrain'] = gather_scores_plain(
    retrain_model, handler_dist, handler_dist.ts_pk_adj)

for name, sc in scores_store.items():
    print(f"{name:<18} adv mean {sc['adv'].mean():7.3f} | "
          f"pos mean {sc['pos'].mean():7.3f} | neg mean {sc['neg'].mean():7.3f}")

In [ ]:
# CVD-safe palette (Okabe-Ito subset), validated: identity is also carried by
# linestyle-consistent mean markers and the legend, never by color alone.
COLORS = {'pos': '#0072B2', 'neg': '#009E73', 'adv': '#D55E00'}
LABELS = {'pos': 'Positive edges', 'neg': 'Negative edges', 'adv': 'Adversarial (unlearned) edges'}
PANELS = ['Before unlearning', 'AIE', 'GAIE', 'Retrain']

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, name in zip(axes.flat, PANELS):
    sc = scores_store.get(name)
    if sc is None:
        ax.set_title(f'{name} (not collected)')
        ax.axis('off')
        continue
    for key in ('pos', 'neg', 'adv'):
        vals = sc[key]
        ax.hist(vals, bins=60, density=True, histtype='step',
                linewidth=2, color=COLORS[key], label=LABELS[key])
        ax.hist(vals, bins=60, density=True, histtype='stepfilled',
                alpha=0.12, color=COLORS[key])
        ax.axvline(float(vals.mean()), color=COLORS[key], linestyle='--', linewidth=1.2)
    ax.set_title(name)
    ax.set_xlabel('Link prediction score')
    ax.set_ylabel('Density')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels_ = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc='upper center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 1.05))
fig.tight_layout()
fig.savefig(f'./logs/score_distributions_{DATASET}.png', dpi=300, bbox_inches='tight')
np.savez(f'./logs/score_distributions_{DATASET}.npz',
         **{f'{n}::{k}': scores_store[n][k] for n in scores_store for k in scores_store[n]})
plt.show()
print(f"Figure saved to ./logs/score_distributions_{DATASET}.png (300 dpi, paper-ready)")

---
## EXP-3 — Efficiency (paper RQ5)

Wall-clock unlearning cost vs. full retraining. Retrain time comes from this session if the
reference was trained here, otherwise take it from the full-run notebook's log/JSON.

In [ ]:
print(f"{'method':<18} {'unlearn(s)':>11} {'finetune(s)':>12} {'total(s)':>9} {'speedup':>9}")
print('-' * 64)
ret_t = retrain_time
if ret_t is None:
    fr = f'./logs/final_results_{args.data}.json'
    if os.path.exists(fr):
        with open(fr) as fs:
            ret_t = json.load(fs)['results'].get('Retrain', {}).get('UnlearnTime')
if ret_t:
    print(f"{'Retrain':<18} {ret_t:>11.1f} {'-':>12} {ret_t:>9.1f} {'1.0x':>9}")
else:
    print(f"{'Retrain':<18} {'N/A (see full-run log)':>34}")

for r in exp_results:
    if r['variant'] != 'full' or not r['adversarial']:
        continue
    tot = r['UnlearnTime'] + r['FinetuneTime']
    sp = f"{ret_t / tot:.1f}x" if ret_t else 'N/A'
    print(f"{r['pipeline'].upper():<18} {r['UnlearnTime']:>11.1f} "
          f"{r['FinetuneTime']:>12.1f} {tot:>9.1f} {sp:>9}")

---
## EXP-4 — Random-request unlearning (privacy scenario)

The deployment scenario: a **clean** backbone must forget a random subset of *genuinely
trained* interactions (right-to-be-forgotten requests), rather than injected attack edges.
The influence encoder is pre-trained on simulated random requests; at evaluation a fresh
random request set is drawn — this tests the encoder's generalization to unseen requests,
matching Algorithm 1 (line 9) of the paper. MI-BF is fully meaningful here too: the dropped
edges were actually learned as positives.

In [ ]:
RUN_RANDOM_SCENARIO = True

if RUN_RANDOM_SCENARIO:
    for p in ('aie', 'gaie'):
        run_variant(p, 'full', adversarial=False, trained_ckpt=CLEAN_CKPT)

    print(f"\n{'pipeline':<8} {'Recall@20':>10} {'NDCG@20':>9} {'MI-BF':>8} {'MI-NG':>8}")
    print('-' * 48)
    for r in exp_results:
        if r['adversarial']:
            continue
        print(f"{r['pipeline']:<8} {r['Recall']:>10.4f} {r['NDCG']:>9.4f} "
              f"{r['MI_BF']:>8.4f} {r['MI_NG']:>8.4f}")

---
## EXP-5 — Loss-weight sensitivity (optional)

Off by default (GPU cost). Enables the hyperparameter-sensitivity figure: MI-NG and Recall
as functions of the unlearning weight (AIE) and the reconstruction weight (GAIE).

In [ ]:
RUN_SENSITIVITY = False

if RUN_SENSITIVITY:
    for w in (0.1, 0.3, 0.5):
        run_variant('aie', f'uw{w}', overrides={'unlearn_wei': w})
    for w in (0.01, 0.03, 0.1):
        run_variant('gaie', f'rw{w}', overrides={'rec_wei': w})

---
## EXP-6 — Multi-seed robustness (optional)

Off by default. For the camera-ready, headline numbers should be mean ± std over >= 3 seeds;
enable and merge the rows (each seed's row is tagged in `variant`).

In [ ]:
RUN_MULTISEED = False
SEEDS = [1234, 2026, 7]

if RUN_MULTISEED:
    for s in SEEDS:
        run_variant('aie', f'full_seed{s}', overrides={'seed': s})
        run_variant('gaie', f'full_seed{s}', overrides={'seed': s})

---
## Export — JSON + LaTeX table sources

In [ ]:
with open(f'./logs/experiments_{DATASET}.json', 'w') as fs:
    json.dump({'dataset': DATASET, 'seed': args.seed, 'adv_method': CFG['adv_method'],
               'results': exp_results}, fs, indent=2)
print(f"All rows saved to ./logs/experiments_{DATASET}.json\n")


def latex_ablation(pipeline):
    lines = [r'\begin{tabular}{lcccc}', r'\toprule',
             r'Variant & Recall@20 & NDCG@20 & MI-BF & MI-NG \\', r'\midrule']
    for r in exp_results:
        if r['pipeline'] != pipeline or not r['adversarial'] or 'seed' in r['variant']:
            continue
        v = r['variant'].replace('_', r'\_')
        lines.append(f"{v} & {r['Recall']:.4f} & {r['NDCG']:.4f} & "
                     f"{r['MI_BF']:.4f} & {r['MI_NG']:.4f} \\\\")
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)


print('% ---- AIE ablation table ----')
print(latex_ablation('aie'))
print()
print('% ---- GAIE ablation table ----')
print(latex_ablation('gaie'))